In [28]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

from scripts.text_matching import normalise_text,remove_diacritics

In [29]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
unmatched_file = Path(project_root / "data/people/unmatched_flat.json")
nopes_file = Path(project_root / "scripts/notebooks/nopes_fixed.json")

In [30]:
with open(people_file , "r") as f:
   people = json.load(f)
with open(variants_file , "r") as f:
   variants = json.load(f)
with open(unmatched_file, "r") as f:
   unmatched = json.load(f)
with open(nopes_file, "r") as f:
   nopes = json.load(f)


In [31]:
unmatched_dict = {}
# rprint(f"unmatched entries: {len(unmatched)}")
total_unmatched_flat = len(unmatched)
total_nopes = len(nopes)

for entry in unmatched:
    composite_id = entry["composite_id"]

    if composite_id not in unmatched_dict:
        unmatched_dict[composite_id] = [entry]
    else:
        unmatched_dict[composite_id].append(entry)

rprint(dict(list(unmatched_dict.items())[:2]))
# rprint(f"len lookup: {len(unmatched_dict)}")


{
    'wien_210_9_10': [
        {
            'composite_id': 'wien_210_9_10',
            'display_name': 'ROMMEL, Otto',
            'sort_order': 1,
            'is_author': True,
            'is_editor': False,
            'is_contributor': False,
            'is_translator': False,
            'is_organisation': False,
            'display_norm': 'rommel, otto',
            'last': 'ROMMEL',
            'first': 'Otto',
            'single': None,
            'original_entry': 'ROMMEL, Otto\nEin Jahrhundert Alt-Wiener Parodie. Herausgegeben von … Wien 
Österreichischer Bundesverlag 1930. 285 S. OLn.',
            'verification_notes': "The entry states 'Herausgegeben von …' but no editor name follows the ellipsis. 
missing_person flag set accordingly. Note: ROMMEL, Otto is listed as the main entry (author/compiler), but the 
'Herausgegeben von …' suggests an editor role may be missing or that Rommel himself is the editor with the name 
omitted by the cataloger.",
            'match_found': False,
            'person_id': None
        }
    ],
    'briefe_45_2_15': [
        {
            'composite_id': 'briefe_45_2_15',
            'display_name': 'Rommel, Otto',
            'sort_order': 2,
            'is_author': False,
            'is_editor': True,
            'is_contributor': False,
            'is_translator': False,
            'is_organisation': False,
            'display_norm': 'rommel, otto',
            'last': 'Rommel',
            'first': 'Otto',
            'single': None,
            'original_entry': "NESTROY, Johann\nSämtliche Werke. Historisch-kritische Gesamtausgabe in zwölf 
Bänden. Herausgegeben von Fritz Brukner und Otto Rommel. Wien Schroll.\nDie Zauberspiele. Erster Teil mit drei 
Bildbeilagen. 1924. XV. 712 S\nDie Zauberspiele. Zweiter Teil mit vier Bildbeilagen. 1924. 772 S.\nDie Parodien. 
Erster Teil mit fünf Bildbeigaben. 1925. 572 S.\nDie Parodien. Zweiter Teil mit neun Bildbeigaben. 1925. 418 
S.\nDie politischen Komödien. Mit neunzehn Bildbeigaben. 1925. 750 S. (ab jetzt 'unter Mitwirkung von Adolf 
Hoffmann')\nDie Volksstücke. Erster Teil. Mit vier Bildbeilagen. 1926. 599 S. (ab jetzt fehlt Zusatz 'in zwölf 
Bänden')\nDie Volksstücke. Zweiter Teil. Mit fünf Bildbeilagen. 1926. 557 S.\nDie Volksstücke. Dritter Teil. Mit 
einer Bildbeilage. 1926. 563 S.\nDie Possen. Erster Teil. Mit dreizehn Bildbeilagen. 1927. 625 S.\nDie Possen. 
Zweiter Teil. Mit sieben Bildbeilagen. 1927. 656 S.\nDie Possen. Dritter Teil. Mit vier Bildbeilagen. 1928. 647 
S.\nDie Possen. Vierter Teil. Mit zwei Bildbeilagen. 1929. 686 S. (ab jetzt fehlt Hoffmann als Hg. Auf letzter 
Seite Nachruf auf den Toten).\nDie Possen. Fünfter Teil. Mit acht Bildbeilagen. 1929. 717 S.\nDie Possen. Sechster 
Teil. Mit vier Bildbeilagen. 1930. 744 S.\nJohann Nestroy. Ein Beitrag zur Geschichte der Wiener Volkskomik von 
Otto Rommel. Mit neunzehn Bildbeigaben. 1930. XVIII, 804 S.\nOriginalleinenbände. Alle Bände verlagsfrisch.",
            'verification_notes': "Die Gesamtausgabe ist auf dem Titelblatt als 'in zwölf Bänden' angekündigt, 
umfasst aber tatsächlich 15 beschriebene Inhaltsbände plus einen abschließenden Kommentarband (Rommel-Monographie),
also 16 Bände insgesamt. Ab Band 6 fehlt der Zusatz 'in zwölf Bänden' auf den Titelblättern. Die Bandnummern sind 
im Eintrag nicht explizit angegeben; die hier vergebenen Nummern 1–16 folgen der Reihenfolge der Beschreibung. Band
16 ist im Eintrag nicht beschrieben — es ist unklar, ob ein 16. Band existiert oder ob die Zählung bei 15 endet. 
Der letzte beschriebene Band (Rommel-Monographie, 1930) könnte als Ergänzungsband außerhalb der Hauptreihe stehen. 
Menschliche Überprüfung der Bandstruktur empfohlen.",
            'match_found': False,
            'person_id': None
        },
        {
            'composite_id': 'briefe_45_2_15',
            'display_name': 'Hoffmann, Adolf',
            'sort_order': 1,
            'is_author': False,
            'is_editor': False,
      

In [ ]:
new_people_complete = {}
people_new = []
b2p_new = []

single_count = 0
proper_single = 0
single_check_count = 0
singles_to_check = {}
nope_processed_count = 0
composite_id_processed = 0


# for name, entry in nopes.items():
for name, entry in list(nopes.items())[:100]:
    nope_processed_count += 1
    last_norm = entry["last_norm"]
    first_norm = entry["first_norm"]
    single_norm = entry["single_norm"]
    family_name = entry["family_name"]
    name_particles = entry["name_particles"]

    # CREATE UNIFIED ID
    if single_norm:
        single_count += 1
        unified_id_base = single_norm

        if last_norm and not family_name:
            proper_single += 1

        if family_name or first_norm:
            single_check_count += 1
            singles_to_check[name] = entry

    elif family_name and not first_norm:
        single_check_count += 1
        singles_to_check[name] = entry

    else:
        first_split = first_norm.split(" ")
        if len(first_split) >= 2:
            first_of_more = first_split[0]
            initials = "_".join(n[0] for n in first_split[1:])
            first_clean = first_of_more + "_" + initials
        else:
            first_clean = first_split[0]

        unified_id_base = last_norm + "_" + first_clean

    unified_id = remove_diacritics(unified_id_base)

    new_people_complete[unified_id] = {
        "unified_id": unified_id,
        "family_name": family_name,
        "given_names": entry["given_names"],
        "name_particles": entry["name_particles"],
        "single_name": entry["single_name"],
        "is_organisation": entry["is_organisation"],
        "entries": []
    }
    # rprint(f"1: people complete, no entries: {new_people_complete}")
    new_people_complete_sorted = dict(sorted(new_people_complete.items()))


    # people_new.append({
    #     "unified_id": unified_id,
    #     "family_name": family_name,
    #     "given_names": entry["given_names"],
    #     "name_particles": entry["name_particles"],
    #     "single_name": entry["single_name"],
    #     "is_organisation": entry["is_organisation"],
    # })
    # rprint(f"2: people list: {people_new}")


    for composite_id in entry["composite_id"]:
        composite_id_processed += 1
        records = unmatched_dict[composite_id]
        for u in records:
            if u["display_norm"] == name:
                display_norm = u["display_norm"]
                display_name = u["display_name"]
                sort_order = u["sort_order"]
                is_author = u["is_author"]
                is_editor = u["is_editor"]
                is_contributor = u["is_contributor"]
                is_translator = u["is_translator"]

                new_people_complete_sorted[unified_id]["entries"].append({
                    "display_name": display_name,
                    "composite_id": composite_id,
                    "sort_order": sort_order,
                    "is_author": is_author,
                    "is_editor": is_editor,
                    "is_contributor": is_contributor,
                    "is_translator": is_translator,
                })
                # rprint(f"3: people complete with entries: {new_people_complete}")
                # break

                # b2p_new.append({
                #     "composite_id": composite_id,
                #     "unified_id": unified_id,
                #     "display_name": display_name,
                #     "family_name": family_name,
                #     "given_names": entry["given_names"],
                #     "name_particles": entry["name_particles"],
                #     "single_name": entry["single_name"],
                #     "sort_order": sort_order,
                #     "is_author": is_author,
                #     "is_editor": is_editor,
                #     "is_contributor": is_contributor,
                #     "is_translator": is_translator,
                # })
                # rprint(f"4: b2p info: {b2p_new}")
                break

# if single_check_count > 0:
#     with open("singles_to_check.json", "w") as f:
#         json.dump(singles_to_check, f, ensure_ascii=False, indent=2)

# with open("people_new_complete.json", "w") as f:
#     json.dump(new_people_complete_sorted, f, ensure_ascii=False, indent=2)

# with open("people_new.json", "w") as f:
#     json.dump(people_new, f, ensure_ascii=False, indent=2)
# with open("b2p_new.json", "w") as f:
#     json.dump(b2p_new, f, ensure_ascii=False, indent=2)


rprint(f"""
    === LOG ===

    total unmatched: {total_unmatched_flat}
    people entries in nopes: {total_nopes}
    singles found: {single_count}
    singles_to_check: {single_check_count}
    people processed: {nope_processed_count}
    new_people_complete count: {len(new_people_complete)}
    people_new count: {len(people_new)}
    composite_ids processed: {composite_id_processed}
    b2p new count: {len(b2p_new)}

    === DONE ===
""")


=== LOG ===

    total unmatched: 1860
    people entries in nopes: 1239
    singles found: 78
    singles_to_check: 0
    people processed: 1239
    new_people_complete count: 1232
    people_new count: 1239
    composite_ids processed: 1371
    b2p new count: 1231

    === DONE ===